# 01 · EDA — four datasets (Mercedes-only)

Goal: filter `merc.csv` (UK/GBP), `germany_dataset.csv` (EUR), `vehicles_craigslist.csv` (US/USD) and `cars-spec-dataset.csv` (spec enrichment) to Mercedes-Benz, profile each, and document the preprocessing each needs before they are pooled into one RM training set. **None of these are Malaysian prices** — they are foreign price levels FX-converted to RM, tagged with `source_market`.

In [ ]:
import sys, pathlib
for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_c / 'app').exists() and (_c / 'ml').exists():
        sys.path.insert(0, str(_c)); break
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from ml import ingest
sns.set_theme(); pd.set_option('display.max_columns', 60)

## merc.csv — UK (GBP)
Already all Mercedes; clean, no nulls; `model` has leading spaces + numeric-only classes (180/200/220/230); mileage is in **miles**.

In [ ]:
merc = pd.read_csv(ingest.MERC_CSV); merc['model'] = merc['model'].str.strip()
print(merc.shape); display(merc.head())
print(merc.isna().sum().to_dict())
display(merc[['transmission','fuelType']].melt().groupby(['variable','value']).size())

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(11,4))
merc['price'].plot.hist(bins=50, ax=ax[0], title='merc price (GBP)')
merc['mileage'].plot.hist(bins=50, ax=ax[1], title='merc mileage (miles)'); plt.tight_layout()

## germany_dataset.csv — Germany (EUR)
Multi-brand -> filter `brand`. Specific model names ('Mercedes-Benz E 220') carry a displacement-hinting badge; `fuel_type` has **leaked garbage** (dates/mileage strings); no engine-size column; mileage already km.

In [ ]:
ger = pd.read_csv(ingest.GERMANY_CSV)
mger = ger[ger['brand'].str.lower().str.contains('mercedes', na=False)]
print('mercedes rows:', mger.shape)
display(mger[['model','year','price_in_euro','transmission_type','fuel_type','mileage_in_km']].head())
print('fuel_type (dirty):', mger['fuel_type'].value_counts().head(8).to_dict())

## vehicles_craigslist.csv — US (USD)
Multi-brand -> filter `manufacturer`. Free-text lowercase model ('e320 cdi', 'ml350', 'benz'); price has 0 -> 3-billion outliers; odometer in **miles**; many nulls; no engine-size column.

In [ ]:
cl = pd.read_csv(ingest.CRAIGSLIST_CSV, low_memory=False)
mcl = cl[cl['manufacturer'].str.lower().str.contains('mercedes', na=False)]
print('mercedes rows:', mcl.shape)
display(mcl[['model','year','price','odometer','transmission','fuel']].head())
print('price describe:', mcl['price'].describe()[['min','50%','max']].to_dict())

## cars-spec-dataset.csv — spec enrichment source
Filter `Company`. ~2.4k Mercedes variants keyed by `Serie` (German) + `Production years`. Strong coverage of the specs we enrich with (parsing handled in `ml.ingest.clean_spec`).

In [ ]:
specs = ingest.clean_spec()
print('parsed Mercedes spec variants:', specs.shape)
cover = {c: f'{specs[c].notna().mean()*100:.0f}%' for c in ingest.ENRICH_NUM + ingest.ENRICH_CAT}
print('enriched-field coverage:', cover)
display(specs.head())

## Class-mapping feasibility (enrichment ceiling)
`ingest.canon_class` maps each source's model string to a canonical class. A row can only be spec-enriched if its class exists in the spec table.

In [ ]:
sc = set(specs['model_class'])
def cov(name, s):
    c = s.map(ingest.canon_class)
    print(f'{name:11s} parsed a class: {c.notna().mean()*100:5.1f}% | '
          f'class in spec: {c.isin(sc).mean()*100:5.1f}%')
cov('merc', merc['model']); cov('germany', mger['model']); cov('craigslist', mcl['model'])

## Preprocessing contract (per dataset)
- **merc (uk):** strip model -> `canon_class`; miles->km; GBP->RM (`fx_gbp_to_rm`); real `engineSize` kept; `source_market='uk'`. Drop numeric-only classes (unmapped).
- **germany:** filter brand; `canon_class`; km as-is; EUR->RM (`fx_eur_to_rm`); normalise dirty `fuel_type`/`transmission`; badge -> engine hint; enrich engine_size.
- **craigslist (us):** filter manufacturer; `canon_class`; miles->km; USD->RM (`fx_usd_to_rm`); drop price/odometer outliers; enrich engine_size.
- **cars-spec:** filter Company; parse displacement/torque/top-speed/accel/boot/gears/aspiration/brakes; build class x year lookup.

Shared pool filters: `1990 <= year <= 2026`, `price_rm in [3k, 2M]`, `mileage <= 500k km`, drop dupes. Then `battery_soh` / `trans_adapt_offset` are engineered (unchanged). All of this is implemented + tested in `ml.ingest`; `02_cleaning.ipynb` runs it.

> **Training-pool decision:** craigslist (US) is *explored here but excluded from the training pool* — its listings are the noisiest (validation MAPE ~49% vs UK 10% / germany 24%). `ml.ingest.build_pool` trains on UK + germany only.